In [6]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [11]:
imgsize = (224, 224)
batchsize = 8
datasetdir = "dataset"

datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15
)

traingen = datagen.flow_from_directory(
    datasetdir,
    target_size=imgsize,
    batch_size=batchsize,
    class_mode="binary",
    subset="training"
)

valgen = datagen.flow_from_directory(
    datasetdir,
    target_size=imgsize,
    batch_size=batchsize,
    class_mode="binary",
    subset="validation"
)

print(f"class indices: {traingen.class_indices}")

Found 88 images belonging to 2 classes.
Found 20 images belonging to 2 classes.
class indices: {'ripe': 0, 'unripe': 1}


In [8]:
basemodel = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

basemodel.trainable = False

model = models.Sequential([
    basemodel,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [12]:
epochs = 10

history = model.fit(
    traingen,
    epochs=epochs,
    validation_data=valgen
)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.6364 - loss: 0.6732 - val_accuracy: 0.7000 - val_loss: 0.6107
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 572ms/step - accuracy: 0.7614 - loss: 0.5123 - val_accuracy: 0.8500 - val_loss: 0.4392
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 553ms/step - accuracy: 0.8750 - loss: 0.3715 - val_accuracy: 0.8000 - val_loss: 0.4846
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 555ms/step - accuracy: 0.8864 - loss: 0.2903 - val_accuracy: 0.8500 - val_loss: 0.3587
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 538ms/step - accuracy: 0.9091 - loss: 0.2740 - val_accuracy: 0.8500 - val_loss: 0.3506
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 542ms/step - accuracy: 0.8864 - loss: 0.2803 - val_accuracy: 0.8500 - val_loss: 0.3302
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 650ms/step - accuracy: 0.9205 - loss: 0.2035 - val_accuracy: 0.8500 - val_loss: 0.2869
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 613ms/step - accuracy: 0.9091 - loss: 0.2001 - val_accuracy: 0.8

In [13]:
model.save('banana_model.keras')